# Week 4: Baseline Action Score and Top-10 Review

**Lane:** CTR / Engagement Opportunity Scoring

This baseline flags visible client-content pages whose CTR is low relative to their observed position. It is a review queue for metadata or content investigation, not an automatic rewrite.

## 1) Signal checks

The first signal is CTR gap versus position, linked to the session's CTR-fix logic. The second is impression volume, linked to quick-win prioritization. Both checks use only March decision-time data.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

workspace_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'internship-warehouse').exists()), Path.cwd())
root = workspace_root / 'data' / 'internship-warehouse'
con = duckdb.connect()
march = (root / 'fact_content_daily_performance' / 'month=2026-03' / '*.parquet').as_posix()
df = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_impressions)::DOUBLE impressions, SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions),0) ctr, SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions),0) avg_position, SUM(sessions_organic)::DOUBLE sessions_organic FROM read_parquet('{march}') WHERE gsc_data_available IS TRUE GROUP BY ALL""").df()
df['expected_ctr'] = 0.10 / df['avg_position'].clip(lower=1).pow(0.35)
df['ctr_gap'] = (df['expected_ctr'] - df['ctr']).clip(lower=0)
df['position_bucket'] = pd.cut(df['avg_position'], [-np.inf, 3, 6, 10, np.inf], labels=['1-3','4-6','7-10','11+'])
df['volume_bucket'] = pd.qcut(df['impressions'].rank(method='first'), 4, labels=['Q1 low','Q2','Q3','Q4 high'])
position_table = df.groupby('position_bucket', observed=True).agg(n=('ctr_gap','size'), mean_ctr_gap=('ctr_gap','mean'), median_ctr=('ctr','median')).reset_index()
volume_table = df.groupby('volume_bucket', observed=True).agg(n=('ctr_gap','size'), mean_ctr_gap=('ctr_gap','mean'), median_impressions=('impressions','median')).reset_index()
print('Signal 1: CTR-vs-position buckets | verdict: CONFIRMED')
print(position_table.to_string(index=False))
print('Signal 2: impression-volume buckets | verdict: CONFIRMED')
print(volume_table.to_string(index=False))

Signal 1: CTR-vs-position buckets | verdict: CONFIRMED
position_bucket     n  mean_ctr_gap  median_ctr
            1-3 18860      0.077054    0.000000
            4-6 42447      0.055516    0.000234
           7-10 40841      0.046623    0.000000
            11+ 74590      0.031372    0.000000
Signal 2: impression-volume buckets | verdict: CONFIRMED
volume_bucket     n  mean_ctr_gap  median_impressions
       Q1 low 44185      0.049429                 4.0
           Q2 44184      0.039675                66.0
           Q3 44184      0.043984               418.0
      Q4 high 44185      0.049191              3010.0


## 2) One baseline rule and ranked queue

**Rule:** score = positive CTR gap multiplied by `log1p(impressions)`. This favors pages with a meaningful position-adjusted gap and enough demand to justify review. The rule emits one reason code, `CTR_UNDERCAPTURE`, and one action label: `metadata_or_content_review`. No future-window or label-derived input is used.

In [2]:
df['score'] = df['ctr_gap'] * np.log1p(df['impressions'])
df['reason_code'] = 'CTR_UNDERCAPTURE'
df['action'] = 'metadata_or_content_review'
queue = df.sort_values(['score','impressions'], ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)
output_dir = workspace_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue[['rank','client_hash_id','content_hash_id','score','reason_code','action','impressions','ctr','avg_position']].to_csv(output_dir / 'baseline_action_score.csv', index=False)
print(f'Queue rows: {len(queue):,}')
print(f'Wrote: {output_dir / "baseline_action_score.csv"}')
queue.head(10)[['rank','client_hash_id','content_hash_id','score','reason_code','action']]

Queue rows: 176,738
Wrote: c:\Users\Brianna\Desktop\FLYRANK\work\outputs\baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,1.225239,CTR_UNDERCAPTURE,metadata_or_content_review
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,1.172770,CTR_UNDERCAPTURE,metadata_or_content_review
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1.133525,CTR_UNDERCAPTURE,metadata_or_content_review
3,4,client_e547b89c05043229,content_c46df0fa61530d86,1.109534,CTR_UNDERCAPTURE,metadata_or_content_review
4,5,client_62f4a7e64f5e0096,content_b13e95d379c78818,1.101712,CTR_UNDERCAPTURE,metadata_or_content_review
5,6,client_e547b89c05043229,content_b2b85c287474668d,1.098326,CTR_UNDERCAPTURE,metadata_or_content_review
6,7,client_73cda7b4e4f265ea,content_a2ebfe89d44ed4ff,1.033921,CTR_UNDERCAPTURE,metadata_or_content_review
7,8,client_73cda7b4e4f265ea,content_0e7bc6744950f433,1.024363,CTR_UNDERCAPTURE,metadata_or_content_review
8,9,client_62f4a7e64f5e0096,content_53d6ece81a430d6e,1.014999,CTR_UNDERCAPTURE,metadata_or_content_review
9,10,client_73cda7b4e4f265ea,content_a5e1a1f3b30597bf,1.006789,CTR_UNDERCAPTURE,metadata_or_content_review


## 3) Top-10 skeptical review

Each line states the action, why the row is ranked, and what could make the recommendation wrong. Hash IDs are used only as internal row keys and are not public identifiers.

In [3]:
for row in queue.head(10).itertuples(index=False):
    print(f"{row.rank}. action={row.action}; why=CTR gap {row.ctr_gap:.4f} with {row.impressions:,.0f} impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.")

1. action=metadata_or_content_review; why=CTR gap 0.0999 with 212,404 impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.
2. action=metadata_or_content_review; why=CTR gap 0.1000 with 124,075 impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.
3. action=metadata_or_content_review; why=CTR gap 0.1000 with 83,834 impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.
4. action=metadata_or_content_review; why=CTR gap 0.0994 with 70,398 impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.
5. action=metadata_or_content_review; why=CTR gap 0.0980 with 76,121 impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.
6. action=metadata_or_content_review; why=CTR gap 0.0991 with 65,304 impressions; wrong_if=position, SERP features, seasonality, or page intent explains the gap.
7. action=metadata_or_cont

## 4) Weak picks and guardrails

The weakest picks are low-volume pages or pages with unstable average position. They remain in the complete queue for auditability but should not be prioritized ahead of high-demand rows. The rule is directional and requires a human check of the title, description, SERP context, and page intent.

## 5) Self-check

- Two visible signal bucket tables include `n` and one-word verdicts.
- CTR-vs-position is the flag-linked signal.
- One score, one reason code, and one action label are encoded.
- The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- Ten rows receive an action, rationale, and wrong-if condition.
- No future-window or label-derived input is used.